# Text/image-to-video test: free open-source video models on Kaggle (T4 ×2)

Can an **Apache-2.0 video model on a free GPU** make short, on-brand clips for the channel (B-roll, animated stills)?
Standalone experiment, sibling of `experiments/t2i-test/`: it doesn't touch the studio pipeline.

**Kaggle:** Accelerator **GPU T4 ×2**, Internet **On**, *Run all* (or *Save & Run All*). Optional secret `HF_TOKEN` (no model here is gated).

| Model | Licence | Mode | Native settings used here |
|---|---|---|---|
| Wan2.2 TI2V-5B | Apache-2.0 | text→video **and image→video** | 480×832, 24 fps |
| Wan2.1 T2V-1.3B | Apache-2.0 | text→video | 480×832, 16 fps |
| CogVideoX-2B | Apache-2.0 | text→video | 720×480 (landscape only), 8 fps |

Left out on licence (non-Apache, custom or revenue-gated): LTX-Video, CogVideoX-5B/1.5. Mochi 1 (Apache) is too big for a T4.

| Step | What | Expected time on T4 ×2 |
|---|---|---|
| 1 | Install + GPU report | ~4 min |
| 2 | Smoke test: one short clip per model, dtype/placement fallback | ~30–60 min (downloads + slow GPU) |
| 3 | 4 **image→video** clips from the t2i-test stills + 2 **text→video** B-roll clips with the winner | ~1–2 h |
| 4 | Zip `t2v-test-results.zip` (clips + `manifest.json`) | ~1 min |


In [ ]:
# 1. Install (newest diffusers from source: Wan 2.2 pipelines need it)
!pip install -q -U git+https://github.com/huggingface/diffusers transformers accelerate sentencepiece ftfy imageio imageio-ffmpeg psutil
# Kaggle ships an old torchao that breaks the newest diffusers' import (FqnToConfig, seen in t2i-test v1). We don't use torchao.
!pip uninstall -y -q torchao
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# Fail fast with the real error if a pipeline class is missing
import importlib, diffusers
for n in ["WanPipeline", "WanImageToVideoPipeline", "AutoencoderKLWan", "CogVideoXPipeline"]:
    assert hasattr(diffusers, n), f"diffusers {diffusers.__version__} has no {n}"; print("ok:", n)

In [ ]:
import os, gc, json, time, platform, zipfile, base64, subprocess, sys, shutil
import numpy as np, torch, transformers, psutil

ON_KAGGLE = os.path.exists("/kaggle")
OUT = "/kaggle/working/t2v" if ON_KAGGLE else "/content/t2v"
for d in ("smoke", "clips", "stills"): os.makedirs(f"{OUT}/{d}", exist_ok=True)
try:
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("No HF_TOKEN secret: continuing anonymously (no model here is gated)")

N_GPU = torch.cuda.device_count()
ENV = dict(platform="kaggle" if ON_KAGGLE else "colab", ram_gb=round(psutil.virtual_memory().total / 2**30, 1),
           gpus=[torch.cuda.get_device_name(i) for i in range(N_GPU)],
           vram_gb=[round(torch.cuda.get_device_properties(i).total_memory / 2**30, 1) for i in range(N_GPU)],
           bf16_native=torch.cuda.is_bf16_supported() if N_GPU else False,
           torch=torch.__version__, diffusers=diffusers.__version__, transformers=transformers.__version__, python=platform.python_version())
print(json.dumps(ENV, indent=1))
assert N_GPU, "No GPU: set the accelerator to GPU T4 x2 and re-run"

## Config: models, house style, shots
Clips are kept short (2–3 s) because a T4 is slow; the test is about **quality and feasibility**, not length. The style prefix is the channel's `reference-flat` look (same as t2i-test).

In [ ]:
MODELS = {
    "wan22-5b":   dict(repo="Wan-AI/Wan2.2-TI2V-5B-Diffusers", t2v="WanPipeline", i2v="WanImageToVideoPipeline", wan_vae=True,
                       w=480, h=832, frames=49, fps=24, steps=30, guidance=5.0, licence="Apache-2.0"),
    "wan21-1.3b": dict(repo="Wan-AI/Wan2.1-T2V-1.3B-Diffusers", t2v="WanPipeline", i2v=None, wan_vae=True,
                       w=480, h=832, frames=33, fps=16, steps=30, guidance=5.0, licence="Apache-2.0"),
    "cogvideox-2b": dict(repo="zai-org/CogVideoX-2b", t2v="CogVideoXPipeline", i2v=None, wan_vae=False,
                       w=720, h=480, frames=49, fps=8, steps=30, guidance=6.0, licence="Apache-2.0"),
}
SMOKE_FRAMES = {"wan22-5b": 33, "wan21-1.3b": 33, "cogvideox-2b": 49}   # short smoke clips (Wan needs 4n+1)
SEED = 1234
PASS_SECONDS, PASS_VRAM_GB = 900, 15     # a 2 s smoke clip in <= 15 min, fits one T4

STYLE = ("Flat vector motion graphics, solid colour blocks, electric blue #2B3BE0, bright yellow #FFD23F, off-white #EEF1F6 "
         "and deep navy #0B1020 only, clean futuristic tech geometry, crisp edges, minimal, modern, high contrast, smooth slow camera "
         "motion, no gradients, no grain, no text, no letters, no logos, no people, not cartoon, not pastel. ")
NEG = "text, letters, watermark, logo, people, faces, hands, blurry, low quality, distorted, flicker, jitter, noisy, gradients"
SMOKE_PROMPT = "A sleek smartphone floating over a perspective grid floor, slowly rotating, a chat bubble rising from its screen."

# image->video: 4 usable stills from t2i-test (embedded below, 480x832), each with a motion prompt
I2V = [("s01", "The smartphone slowly turns while the chat bubble rises and gently pulses; slow camera push-in."),
       ("s04", "The ribbon of tiles scrolls steadily into the distance; smooth forward dolly along the ribbon."),
       ("s05", "The camera glides slowly down the aisle between the shelves of tiles; tiles glint as it passes."),
       ("s08", "Small blue tiles keep dropping into the glass meter and the gauge needle climbs; gentle push-in.")]
# text->video B-roll, landscape 16:9 for long-form (Wan 832x480)
T2V = [("b1", "A glowing envelope icon flies across a flat blue network grid; nodes light up one after another along its path."),
       ("b2", "A smartphone lies on a flat navy surface; a bold red warning shield pops up above its screen and pulses.")]
STILLS = {"s01": "...", "s04": "...", "s05": "...", "s08": "..."}
for sid, b64 in STILLS.items():
    open(f"{OUT}/stills/{sid}.jpg", "wb").write(base64.b64decode(b64))
print("models:", list(MODELS), "| stills:", sorted(STILLS))

## Helpers: every attempt runs in its own child process
Lessons from t2i-test: an out-of-memory or `Killed` attempt must not take the notebook down, so each (model, dtype, placement) attempt is
its own process. **fp16 goes first**: a T4 has no native bf16 (it is emulated and much slower); bf16 is the fallback when fp16 gives
NaN/black frames. Wan's VAE always runs in fp32 (model card).

Ladder on T4 ×2: fp16 split across both GPUs → bf16 split → fp16 CPU-offload → bf16 CPU-offload.

In [ ]:
%%writefile /tmp/t2v_worker.py
# One attempt = one process: python t2v_worker.py <smoke|i2v|t2v> <model> <dtype> <placement> <out_dir>
import sys, json, time, numpy as np, torch, diffusers as D
from diffusers.utils import export_to_video, load_image
mode, name, dtype_s, placement, out = sys.argv[1:6]
C = json.load(open(f"{out}/config.json")); cfg = C["MODELS"][name]; dtype = getattr(torch, dtype_s)

def load(kind):
    cls = getattr(D, cfg[kind])
    kw = dict(torch_dtype=dtype, low_cpu_mem_usage=True)
    if cfg["wan_vae"]:
        kw["vae"] = D.AutoencoderKLWan.from_pretrained(cfg["repo"], subfolder="vae", torch_dtype=torch.float32)
    if placement == "balanced": kw["device_map"] = "balanced"
    pipe = cls.from_pretrained(cfg["repo"], **kw)
    if placement == "offload": pipe.enable_model_cpu_offload()
    if placement == "gpu": pipe.to("cuda")
    if placement == "balanced" and cfg["wan_vae"]:
        dev = next(pipe.transformer.parameters()).device; pipe.vae.to(dev)   # pre-loaded VAE: put it next to the transformer
    for f in ("enable_tiling", "enable_slicing"):
        if hasattr(pipe.vae, f): getattr(pipe.vae, f)()
    pipe.set_progress_bar_config(disable=True)
    return pipe

def run(pipe, prompt, frames, w, h, image=None):
    kw = dict(prompt=C["STYLE"] + prompt, negative_prompt=C["NEG"], num_frames=frames, num_inference_steps=cfg["steps"],
              guidance_scale=cfg["guidance"], generator=torch.Generator("cpu").manual_seed(C["SEED"]), output_type="np")
    if image is not None: kw["image"] = image
    kw["width"], kw["height"] = w, h
    torch.cuda.synchronize(); t = time.time()
    v = pipe(**kw).frames[0]
    torch.cuda.synchronize(); return np.asarray(v), round(time.time() - t, 1)

def bad(v):
    return bool(np.isnan(v).any() or float(v.std()) < 0.01)   # NaN / black / flat (frames are 0..1 floats)

def save(v, path, fps):
    export_to_video(list(np.clip(v, 0, 1)), path, fps=fps)

def peak():
    return round(max(torch.cuda.max_memory_allocated(i) for i in range(torch.cuda.device_count())) / 2**30, 1)

if mode == "smoke":
    t = time.time(); pipe = load("t2v"); load_s = round(time.time() - t, 1)
    fr = C["SMOKE_FRAMES"][name]; v, secs = run(pipe, C["SMOKE_PROMPT"], fr, cfg["w"], cfg["h"])
    p = f"{out}/smoke/{name}.mp4"; save(v, p, cfg["fps"])
    print("RESULT " + json.dumps(dict(seconds_per_clip=secs, frames=fr, clip_seconds=round(fr / cfg["fps"], 2), load_s=load_s,
                                      peak_vram_gb=peak(), black_or_nan=bad(v), video=f"smoke/{name}.mp4")))
else:
    jobs = C["I2V"] if mode == "i2v" else C["T2V"]
    pipe = load("i2v" if mode == "i2v" else "t2v"); res = []
    for jid, prompt in jobs:
        if mode == "i2v":
            img = load_image(f"{out}/stills/{jid}.jpg"); w, h = cfg["w"], cfg["h"]
        else:
            img = None; w, h = cfg["h"], cfg["w"]              # B-roll is landscape: swap to 832x480
        try:
            v, secs = run(pipe, prompt, cfg["frames"], w, h, img)
            p = f"{out}/clips/{mode}-{jid}.mp4"; save(v, p, cfg["fps"])
            res.append(dict(id=jid, mode=mode, prompt=prompt, seconds=secs, size=[w, h], frames=cfg["frames"], fps=cfg["fps"],
                            black_or_nan=bad(v), video=f"clips/{mode}-{jid}.mp4"))
        except Exception as e:
            res.append(dict(id=jid, mode=mode, error=repr(e)[:400]))
        print(f"done {jid}", res[-1].get("seconds", res[-1].get("error")), flush=True)
    print("RESULT " + json.dumps(dict(clips=res, peak_vram_gb=peak())))


In [ ]:
json.dump(dict(MODELS=MODELS, SMOKE_FRAMES=SMOKE_FRAMES, SEED=SEED, STYLE=STYLE, NEG=NEG, SMOKE_PROMPT=SMOKE_PROMPT,
               I2V=I2V, T2V=T2V, OUT=OUT), open(f"{OUT}/config.json", "w"))

def run_worker(mode, name, dtype, placement, timeout=3600):
    # (result, error): a Killed/OOM child only loses that one attempt
    try:
        p = subprocess.run([sys.executable, "/tmp/t2v_worker.py", mode, name, dtype, placement, OUT],
                           capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        return None, f"timeout after {timeout}s"
    for l in p.stdout.splitlines():
        if l.startswith("done "): print("   ", l)
    lines = [l for l in p.stdout.splitlines() if l.startswith("RESULT ")]
    if p.returncode == 0 and lines: return json.loads(lines[-1][7:]), None
    if p.returncode in (-9, 137): return None, "Killed (out of system RAM)"
    tail = [l for l in p.stderr.strip().splitlines() if l.strip()][-6:]
    return None, f"exit {p.returncode}: " + " | ".join(tail)[-700:]

def ladder():
    if N_GPU >= 2: return [("float16", "balanced"), ("bfloat16", "balanced"), ("float16", "offload"), ("bfloat16", "offload")]
    return [("float16", "offload"), ("bfloat16", "offload")]
print("ladder:", ladder())

In [ ]:
# 2. Smoke test: one short text->video clip per model; first ladder rung that works AND isn't black/NaN wins.
#    Saved after every model; if anything dies, re-run this cell and finished models are skipped.
SMOKE_JSON = f"{OUT}/smoke/smoke.json"
smoke = json.load(open(SMOKE_JSON))["smoke"] if os.path.exists(SMOKE_JSON) else []
done = {r["model"] for r in smoke}
for name, cfg in MODELS.items():
    if name in done: print("SKIP", name, "(done)"); continue
    rec = dict(model=name, repo=cfg["repo"], licence=cfg["licence"], attempts=[])
    for dtype, placement in ladder():
        t0 = time.time(); res, err = run_worker("smoke", name, dtype, placement)
        att = dict(dtype=dtype, placement=placement, wall_s=round(time.time() - t0), **(res or {}), **({"error": err} if err else {}))
        rec["attempts"].append(att)
        if err: print(f"FAIL {name:13s} {dtype:8s} {placement:8s} {err[:400]}"); continue
        if res["black_or_nan"]: print(f"BAD  {name:13s} {dtype:8s} {placement:8s} black/NaN clip, next rung"); continue
        att["pass"] = res["seconds_per_clip"] <= PASS_SECONDS and res["peak_vram_gb"] <= PASS_VRAM_GB
        rec["result"] = att
        print(f"OK   {name:13s} {dtype:8s} {placement:8s} {res['seconds_per_clip']:7.1f}s for {res['clip_seconds']}s clip  "
              f"load {res['load_s']}s  peak {res['peak_vram_gb']} GB  pass={att['pass']}")
        break
    smoke.append(rec); json.dump(dict(env=ENV, smoke=smoke), open(SMOKE_JSON, "w"), indent=1)
    if name != "wan22-5b":   # free disk; the 5B is reused in step 3
        shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub/models--" + cfg["repo"].replace("/", "--")), ignore_errors=True)

## 3. Clips with the winner
Image→video needs Wan2.2-5B (the only model here with an image input). If it passed the smoke test it's used for both sets;
otherwise image→video is skipped and the fastest passing model makes the B-roll.

In [ ]:
res_by = {r["model"]: r.get("result") for r in smoke}
clips = []
w5 = res_by.get("wan22-5b")
if w5:
    r, err = run_worker("i2v", "wan22-5b", w5["dtype"], w5["placement"], timeout=4 * 3600)
    print("i2v:", "ok" if r else err); clips += (r or {}).get("clips", [])
passed = [m for m, r in res_by.items() if r and r.get("pass")]
BROLL = "wan22-5b" if w5 else (min(passed, key=lambda m: res_by[m]["seconds_per_clip"]) if passed else None)
if BROLL:
    b = res_by[BROLL]; r, err = run_worker("t2v", BROLL, b["dtype"], b["placement"], timeout=3 * 3600)
    print("t2v:", BROLL, "ok" if r else err); clips += (r or {}).get("clips", [])
for c in clips: print(c.get("id"), c.get("mode"), c.get("seconds"), "black/NaN" if c.get("black_or_nan") else "", c.get("error", ""))

In [ ]:
# 4. Manifest + zip (download it, unzip into experiments/t2v-test/results/)
ok = [c for c in clips if "seconds" in c]
manifest = dict(env=ENV, smoke=smoke, broll_model=BROLL, clips=clips,
                avg_seconds_per_clip=round(sum(c["seconds"] for c in ok) / len(ok), 1) if ok else None)
json.dump(manifest, open(f"{OUT}/manifest.json", "w"), indent=1)
zpath = f"{OUT}/../t2v-test-results.zip"
with zipfile.ZipFile(zpath, "w") as z:
    for root, _, files in os.walk(OUT):
        for f in files: z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), OUT))
print("zip:", zpath, round(os.path.getsize(zpath) / 2**20, 1), "MB")
print(json.dumps({k: v for k, v in manifest.items() if k not in ("smoke", "clips")}, indent=1))